# Challenge: Overfitting on Other Datasets

## Download data from `yfinance`

In [104]:
import yfinance as yf

ticker = 'META'
df = yf.download(ticker, multi_level_index=False, auto_adjust=False)
df

[*********************100%***********************]  1 of 1 completed


,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2026-04-30,611.909973,611.909973,620.849976,600.000000,619.320007,52765000
2026-05-01,608.750000,608.750000,618.880005,606.109985,614.690002,21404000
2026-05-04,610.409973,610.409973,614.000000,602.750000,607.940002,16203900
2026-05-05,604.960022,604.960022,614.349976,600.359985,613.309998,17171300
2026-05-06,612.880005,612.880005,619.950012,598.099976,601.049988,19915500
2026-05-07,616.809998,616.809998,624.979980,613.539978,614.719971,12307400
2026-05-08,609.630005,609.630005,616.770020,606.059998,615.200012,13557000
2026-05-11,598.859985,598.859985,604.909973,598.080017,604.570007,15940600
2026-05-12,603.000000,603.000000,603.750000,592.599976,594.820007,11351600


## Preprocess the data

### Filter the date range

- Since 1 year ago at least

In [105]:
df = df.loc['2020-01-01':].copy()

### Create the target variable

#### Percentage change

- Percentage change on `Adj Close` for tomorrow

In [106]:
df['change_tomorrow'] = df['Adj Close'].pct_change(-1)
df.change_tomorrow = df.change_tomorrow * -1
df.change_tomorrow = df.change_tomorrow * 100

#### Remove rows with any missing data

In [107]:
df = df.dropna().copy()
df

,Adj Close,Close,High,Low,Open,Volume,change_tomorrow
Date,,,,,,,
2026-04-30,611.909973,611.909973,620.849976,600.000000,619.320007,52765000,-0.519092
2026-05-01,608.750000,608.750000,618.880005,606.109985,614.690002,21404000,0.271944
2026-05-04,610.409973,610.409973,614.000000,602.750000,607.940002,16203900,-0.900878
2026-05-05,604.960022,604.960022,614.349976,600.359985,613.309998,17171300,1.292257
2026-05-06,612.880005,612.880005,619.950012,598.099976,601.049988,19915500,0.637148
2026-05-07,616.809998,616.809998,624.979980,613.539978,614.719971,12307400,-1.177762
2026-05-08,609.630005,609.630005,616.770020,606.059998,615.200012,13557000,-1.798420
2026-05-11,598.859985,598.859985,604.909973,598.080017,604.570007,15940600,0.686570
2026-05-12,603.000000,603.000000,603.750000,592.599976,594.820007,11351600,2.210402


## Machine Learning modelling

### Feature selection

1. Target: which variable do you want to predict?
2. Explanatory: which variables will you use to calculate the prediction?

In [108]:
y = df.change_tomorrow
X = df.drop(columns='change_tomorrow')

### Train test split

In [109]:
from sklearn.model_selection import train_test_split

In [110]:
train_test_split

<function sklearn.model_selection._split.train_test_split(*arrays, test_size=None, train_size=None, random_state=None, shuffle=True, stratify=None)>

In [111]:
X_train, X_test, y_train, y_test = train_test_split(
...     X, y, test_size=0.33, random_state=42)

### Fit the model on train set

In [112]:
from sklearn.tree import DecisionTreeRegressor

In [113]:
model_dt_split= DecisionTreeRegressor(max_depth=15,random_state=42 )

In [114]:
model_dt_split.fit(X=X_train, y=y_train)

DecisionTreeRegressor(max_depth=15, random_state=42)

### Evaluate model

#### On test set

In [115]:
from sklearn.metrics import mean_squared_error

y_pred_test= model_dt_split.predict(X=X_test)
mean_squared_error(y_true=y_test,y_pred=y_pred_test)

5.251959156890711

#### On train set

In [116]:
X_train, X_test, y_train, y_test = train_test_split(
...     X, y, test_size=0.33, random_state=42)

## Backtesting

In [117]:
import pandas as pd

In [118]:
from backtesting import Backtest, Strategy

### Create the `Strategy`

In [119]:
class Regression(Strategy):
    limit_buy = 1
    limit_sell = -5
    
    def init(self):
        self.model = DecisionTreeRegressor(max_depth=15, random_state=42)
        self.already_bought = False
        
        self.model.fit(X= X_train, y=y_train)

    def next(self):
        explanatory_today = self.data.df.iloc[[-1], :]
        forecast_tomorrow = self.model.predict(explanatory_today)[0]
        
        if forecast_tomorrow > self.limit_buy and self.already_bought == False:
            self.buy()
            self.already_bought = True
        elif forecast_tomorrow < self.limit_sell and self.already_bought == True:
            self.sell()
            self.already_bought = False
        else:
            pass

### Run the backtest on `test` data

In [120]:
bt_test = Backtest(X_test, Regression,
              cash=10000, commission=.002, exclusive_orders=True)

/var/folders/rw/hsc4g_3s03n19_15p1x0hn7m0000gn/T/ipykernel_28994/1373011202.py:1: UserWarning: Data index is not sorted in ascending order. Sorting.
  bt_test = Backtest(X_test, Regression,


In [121]:
results = bt_test.run(limit_buy=1, limit_sell=-5)

df_results_test = results.to_frame(name='Values').loc[:'Return [%]']\
    .rename({'Values':'Out of Sample (Test)'}, axis=1)
df_results_test

,Out of Sample (Test)
Start,2026-04-30 00:00:00
End,2026-05-26 00:00:00
Duration,26 days 00:00:00
Exposure Time [%],0.0
Equity Final [$],10000.0
Equity Peak [$],10000.0
Return [%],0.0


### Run the backtest on `train` data

In [122]:
bt_train = Backtest(X_train, Regression,
              cash=10000, commission=.002, exclusive_orders=True)

results = bt_train.run(limit_buy=1, limit_sell=-5)

df_results_train = results.to_frame(name='Values').loc[:'Return [%]']\
    .rename({'Values':'In Sample (Train)'}, axis=1)
df_results_train

/var/folders/rw/hsc4g_3s03n19_15p1x0hn7m0000gn/T/ipykernel_28994/3486511487.py:1: UserWarning: Data index is not sorted in ascending order. Sorting.
  bt_train = Backtest(X_train, Regression,


,In Sample (Train)
Start,2026-05-04 00:00:00
End,2026-05-28 00:00:00
Duration,24 days 00:00:00
Exposure Time [%],84.615385
Equity Final [$],10595.966596
Equity Peak [$],10595.966596
Return [%],5.959666


### Compare both backtests

- HINT: Concatenate the previous `DataFrames`

In [123]:
df_results= pd.concat([df_results_test,df_results_test], axis =1)
df_results

,Out of Sample (Test),Out of Sample (Test)
Start,2026-04-30 00:00:00,2026-04-30 00:00:00
End,2026-05-26 00:00:00,2026-05-26 00:00:00
Duration,26 days 00:00:00,26 days 00:00:00
Exposure Time [%],0.0,0.0
Equity Final [$],10000.0,10000.0
Equity Peak [$],10000.0,10000.0
Return [%],0.0,0.0


#### Plot both backtest reports

In [124]:
bt_test.plot(filename='reports_backtesting/regression_test_set.html')
bt_train.plot(filename='reports_backtesting/regression_train_set.html')

/Users/neemaurassa/Library/Python/3.9/lib/python/site-packages/backtesting/_plotting.py:250: UserWarning: DatetimeFormatter scales now only accept a single format. Using the first provided: '%d %b'
  formatter=DatetimeTickFormatter(days=['%d %b', '%a %d'],
/Users/neemaurassa/Library/Python/3.9/lib/python/site-packages/backtesting/_plotting.py:250: UserWarning: DatetimeFormatter scales now only accept a single format. Using the first provided: '%m/%Y'
  formatter=DatetimeTickFormatter(days=['%d %b', '%a %d'],
/Users/neemaurassa/Library/Python/3.9/lib/python/site-packages/backtesting/_plotting.py:455: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df2 = (df.assign(_width=1).set_index('datetime')
/Users/neemaurassa/Library/Python/3.9/lib/python/site-packages/backtesting/_plotting.py:659: UserWarning: found multiple competing values for 'toolbar.active_drag' property; using the latest value
  fig = gridplot(
/Users/neemaurassa/Library/P

GridPlot(id='p1838', ...)

## How to solve the overfitting problem?

> Walk Forward Validation as a realistic approach to backtesting.

Next tutorial → [Walk Forward Validation]()

![](<src/10_Table_Validation Methods.png>)